<a href="https://colab.research.google.com/github/AhsanMalik0/Practice-Portfolio/blob/main/OCR_test_Urdu.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import string
import numpy as np
from PIL import Image, ImageOps
import tensorflow as tf
from tensorflow.keras import layers, mixed_precision
from keras import regularizers
import matplotlib.pyplot as plt
import string
from tensorflow import keras



In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("drsaadbinahmed/unhd-dataset")

print("Path to dataset files:", path)

100%|██████████| 820M/820M [00:07<00:00, 117MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/drsaadbinahmed/unhd-dataset/versions/1


In [ ]:
dataset_path = os.path.join(path,os.listdir(path)[0])

In [ ]:
# char_list = string.digits + 'OTDB?'  # 0-9, A, D, O, T, space
# char_to_num = {char: i+3 for i, char in enumerate(char_list)}  # Start from 3

# # Special tokens
# START_TOKEN = 0
# END_TOKEN = 1
# PADDING_TOKEN = 2

# # Add special tokens to conversion
# num_to_char = {i+3: char for char, i in zip(char_list, range(len(char_list)))}
# num_to_char[START_TOKEN] = '<START>'
# num_to_char[END_TOKEN] = '<END>'
# num_to_char[PADDING_TOKEN] = '<PAD>'

# # Vocabulary size includes all characters + special tokens
# num_classes = len(char_list) + 3  # chars + START + END + PAD

# # Image and sequence parameters
TARGET_HEIGHT = 64
TARGET_WIDTH = 512
MAX_LABEL_LEN = 64
# print(f"Vocabulary size: {num_classes}")
# print(f"Character mapping: {char_to_num}")
# print(f"Image size: {TARGET_HEIGHT}x{TARGET_WIDTH}")

In [ ]:
files = os.listdir(dataset_path)
images = [i for i in files if i[-4:]=='.png']
label = [i for i in files if i[-7:]=='.gt.txt']
# Create a set of image base names (without the extension)
image_base_names = {i[:-4] for i in images}

# Create a set of label base names (without the extension)
label_base_names = {i[:-7] for i in label}

# Images without labels
images_without_labels = image_base_names - label_base_names

# Labels without corresponding images
labels_without_images = label_base_names - image_base_names

print(f"Images without labels: {images_without_labels}")
print(f"Labels without images: {labels_without_images}")

Images without labels: set()
Labels without images: set()


In [ ]:
import os
import shutil

# Define source directory (where your files are currently stored)
source_dir = dataset_path

# Define destination directories for images without labels and labels without images
images_no_label_dir = '/content/images_without_labels'
labels_no_image_dir = '/content/labels_without_images'

# Ensure the destination directories exist
os.makedirs(images_no_label_dir, exist_ok=True)
os.makedirs(labels_no_image_dir, exist_ok=True)

# List of image and label file names
images = [i for i in os.listdir(source_dir) if i.endswith('.png')]
labels = [i for i in os.listdir(source_dir) if i.endswith('.gt.txt')]

# Create a set of image base names (without the extension)
image_base_names = {i[:-4] for i in images}

# Create a set of label base names (without the extension)
label_base_names = {i[:-7] for i in labels}

# Images without labels
images_without_labels = image_base_names - label_base_names

# Labels without corresponding images
labels_without_images = label_base_names - image_base_names

# Move images without labels to the specified folder
for image in images:
    base_name = image[:-4]
    if base_name in images_without_labels:
        source_path = os.path.join(source_dir, image)
        dest_path = os.path.join(images_no_label_dir, image)
        shutil.move(source_path, dest_path)
        print(f"Moved image without label: {image}")

# Move labels without images to the specified folder
for label in labels:
    base_name = label[:-7]
    if base_name in labels_without_images:
        source_path = os.path.join(source_dir, label)
        dest_path = os.path.join(labels_no_image_dir, label)
        shutil.move(source_path, dest_path)
        print(f"Moved label without image: {label}")

In [ ]:
!apt-get update
!apt-get install -y imagemagick
!mogrify -density 300 /root/.cache/kagglehub/datasets/drsaadbinahmed/unhd-dataset/versions/1/UNHD-Complete-Data/*.png

Hit:1 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:2 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:3 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:4 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:5 https://cli.github.com/packages stable InRelease
Hit:6 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:7 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:8 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:10 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading packag

In [ ]:
# Load model directly
from transformers import AutoTokenizer, AutoModelForMaskedLM

tokenizer = AutoTokenizer.from_pretrained("urduhack/roberta-urdu-small")
# model = AutoModelForMaskedLM.from_pretrained("urduhack/roberta-urdu-small")

In [ ]:
vocab_size = tokenizer.vocab_size
vocab_size = vocab_size + 1
MAX_LABEL_LEN = 64

In [ ]:
NON_JOINING_URDU = set("اآدڈذرڑزژو")
SEPARATORS = set("،؛؟۔,;:.!?()[]{}«»\"'-_/\\|@#$%&*")
URDU_DIGITS = set("۰۱۲۳۴۵۶۷۸۹")

# Combine them into one set
NON_JOINING_URDU.update(URDU_DIGITS)
def split_by_shape(word):
    parts = []
    current = ""
    for ch in word:
        # If character is a separator
        if ch in SEPARATORS:
            if current:
                parts.append(current)
                current = ""
            parts.append(ch)
            continue
        # Normal joining logic
        if not current:
            current = ch
        else:
            if current[-1] in NON_JOINING_URDU:
                parts.append(current)
                current = ch
            else:
                current += ch
    if current:
        parts.append(current)

    return parts


In [ ]:
# ============= IMAGE PREPROCESSING =============
def prepare_image(image, target_height=TARGET_HEIGHT, target_width=TARGET_WIDTH):
    width, height = image.size

    # Calculate scaling factor to fit within target while preserving aspect ratio
    scale = min(target_width / width, target_height / height)

    if scale < 1.0:  # Only resize if image is larger
        new_width = int(width * scale)
        new_height = int(height * scale)
        image = image.resize((new_width, new_height), Image.LANCZOS)
        width, height = new_width, new_height

    # Create a white canvas of target size
    canvas = Image.new('L', (target_width, target_height), 255)

    # Paste the image in the center
    paste_x = (target_width - width) // 2
    paste_y = (target_height - height) // 2
    canvas.paste(image, (paste_x, paste_y))

    return canvas

In [ ]:
# ============= Single IMAGE PREPROCESSING =============

def preprocess_image(image_path, target_height=TARGET_HEIGHT, target_width=TARGET_WIDTH):

    # Load image
    image = Image.open(image_path).convert("L")
    print(True)

    # Resize/pad
    image = prepare_image(image, target_height, target_width)
    print(False)

    # Convert to numpy and normalize
    image_np = np.array(image).astype(np.float32)

    # Normalize to [0, 1]
    image_np = image_np / 255.0

    # Optional: Invert if text is dark on light background
    # image_np = 1.0 - image_np

    # Add channel dimension: (H, W, 1)
    image_np = np.expand_dims(image_np, axis=-1)

    return image_np

In [ ]:
def encode_label(label_text):
  tokens = tokenizer(
    label_text,
    padding="max_length",
    truncation=True,
    max_length=64,   # fixed length
    return_tensors="tf"
    )
  input_ids = tokens["input_ids"]
  # input_ids = tf.where(
  #     input_ids == tokenizer.pad_token_id,
  #     vocab_size+1,
  #     input_ids
  # )
  return input_ids

In [ ]:
vocab_size

52001

In [ ]:
def decode_label(token_ids, remove_special=True):
  decoded_text = tokenizer.decode(token_ids, skip_special_tokens=True)
  return decoded_text

In [ ]:
# ============= DATASET PREPARATION =============
def prepare_data(data_dir, img_height=TARGET_HEIGHT, img_width=TARGET_WIDTH,
                 max_label_len=MAX_LABEL_LEN, verbose=True):
    image_files = [f for f in os.listdir(data_dir) if f.lower().endswith(".png")]

    if len(image_files) == 0:
        raise ValueError(f"No .jpg files found in {data_dir}")

    images = []
    labels = []
    skipped = 0

    for img_file in image_files:
        img_path = os.path.join(data_dir, img_file)
        label_path = os.path.splitext(img_path)[0] + ".gt.txt"

        # Check if label file exists
        if not os.path.exists(label_path):
            print(f"Warning: Label file not found for {img_file}, skipping")
            skipped += 1
            continue

        try:
            # Load and preprocess image
            image_np = preprocess_image(img_path, img_height, img_width)

            # Load label
            with open(label_path, "r") as f:
                label_text = f.readline().strip()
            tokens = []
            for word in label_text.split():
              tokens.extend(split_by_shape(word))

            # Skip empty labels
            if not label_text:
                print(f"Warning: Empty label for {img_file}, skipping")
                skipped += 1
                continue

            # Encode label (with START and END tokens)
            label_seq = encode_label(" ".join(tokens))

            if verbose:
                print(f"Loaded: {img_file}")
                print(f"  Label text: '{" ".join(tokens)}'")
                print(f"  Label tokens: {label_seq}")
                print(f"  Decoded: '{decode_label(label_seq[0])}'")

            images.append(image_np)
            labels.append(label_seq[0])

        except Exception as e:
            print(f"Error processing {img_file}: {str(e)}")
            skipped += 1
            continue

    if len(images) == 0:
        raise ValueError("No valid images loaded!")

    # Pad labels to max length
    # padded_labels = tf.keras.preprocessing.sequence.pad_sequences(
    #     labels,
    #     maxlen=max_label_len,
    #     padding='post',
    #     value=PADDING_TOKEN
    # )

    print(f"\n{'='*60}")
    print(f"Dataset prepared:")
    print(f"  Total files: {len(image_files)}")
    print(f"  Successfully loaded: {len(images)}")
    print(f"  Skipped: {skipped}")
    print(f"  Image shape: {images[0].shape}")
    # print(f"  Label shape: {padded_labels.shape}")
    # print(f"  Label range: [{padded_labels.min()}, {padded_labels.max()}]")
    print(f"{'='*60}\n")

    return np.array(images), np.array(labels)

In [ ]:
# ============= DATA VALIDATION =============
def validate_dataset(images, labels):

    print("Validating dataset...")

    # Check shapes
    assert len(images) == len(labels), "Mismatch between images and labels"
    assert images.ndim == 4, f"Images should be 4D, got {images.ndim}D"
    assert labels.ndim == 2, f"Labels should be 2D, got {labels.ndim}D"

    # Check value ranges
    assert images.min() >= 0 and images.max() <= 1, \
        f"Images should be normalized to [0,1], got [{images.min()}, {images.max()}]"

    assert labels.min() >= 0 and labels.max() < vocab_size, \
        f"Labels should be in [0, {vocab_size}), got [{labels.min()}, {labels.max()}]"

    # Check for valid sequences
    for i, label in enumerate(labels[:10]):  # Check first 10
        # Remove padding
        label_no_pad = label[label != tokenizer.pad_token_id]

        # Check if has START and END
        if len(label_no_pad) > 0:
            if label_no_pad[0] != tokenizer.bos_token_id:
                print(f"Warning: Label {i} doesn't start with START_TOKEN")

            # Find first END token
            end_indices = np.where(label_no_pad == END_TOKEN)[0]
            if len(end_indices) == 0:
                print(f"Warning: Label {i} doesn't have END_TOKEN")

    # Print statistics
    label_lengths = np.sum(labels != tokenizer.pad_token_id, axis=1)
    print(f"\nLabel length statistics:")
    print(f"  Min: {label_lengths.min()}")
    print(f"  Max: {label_lengths.max()}")
    print(f"  Mean: {label_lengths.mean():.2f}")
    print(f"  Median: {np.median(label_lengths):.2f}")

    print("\nSample labels (first 5):")
    for i in range(min(5, len(labels))):
        label_no_pad = labels[i][labels[i] != tokenizer.pad_token_id]
        decoded = decode_label(label_no_pad)
        print(f"  {i}: {label_no_pad} -> '{decoded}'")

    print("\n✓ Dataset validation passed!")

In [ ]:
# ============= USAGE EXAMPLE =============
if __name__ == "__main__":
    # Load data
    data_dir = dataset_path#"/home/green-fin/pythonfiles/Prediction/train/MICR"

    images, t_labels = prepare_data(data_dir, verbose=True)

    # Validate
    # validate_dataset(images, labels)

    # Save configuration for later use
    # config = {
    #     'num_classes': num_classes,
    #     # 'padding_token': PADDING_TOKEN,
    #     # 'start_token': START_TOKEN,
    #     # 'end_token': END_TOKEN,
    #     'max_label_len': MAX_LABEL_LEN,
    #     'img_height': TARGET_HEIGHT,
    #     'img_width': TARGET_WIDTH,
    #     'char_to_num': char_to_num,
    #     'num_to_char': num_to_char,
    # }

    # print("\nConfiguration for training:")
    # print(f"  num_classes = {num_classes}")
    # # print(f"  padding_token = {PADDING_TOKEN}")
    # # print(f"  start_token = {START_TOKEN}")
    # # print(f"  end_token = {END_TOKEN}")
    # print(f"  max_length = {MAX_LABEL_LEN}")

    # Ready for training!
    print("\n✓ Data ready for training!")
    print(f"Use: train_model(images, labels, encoder, decoder, ...)")

Streaming output truncated to the last 5000 lines.
  Label text: 'ا ن کے جا نو ر مر ر ہےہیں ۔ یو ں تو بر یتا نیہ کے ا کثر علا قو ں میں مو یشی'
  Label tokens: [[    0   261   287   292   532   569   312   878   312   316  2332   300
   1599  1638   418   520   360   411 27693   292   277 34227  7537   904
   1638   304   427 28668     2     1     1     1     1     1     1     1
      1     1     1     1     1     1     1     1     1     1     1     1
      1     1     1     1     1     1     1     1     1     1     1     1
      1     1     1     1]]
  Decoded: 'ا ن کے جا نو ر مر ر ہےہیں ۔ یو ں تو بر یتا نیہ کے ا کثر علا قو ں میں مو یشی'
True
False
Loaded: 299_12.png
  Label text: 'یہ جا نیے کہ ہر قسم کے بچے کے کھلو نے کیسے صا ف کر یں ۔'
  Label tokens: [[    0   491   532   678   269   313   680  1363   292  1819   292 12297
    330  3008 16386   346   309  6943   300     2     1     1     1     1
      1     1     1     1     1     1     1     1     1     1     1     1
      1     1 

In [ ]:
t_labels[4]

array([    0,   261,   287,   714,   267,  1638,   304,   326,   316,
         328, 16784,  1638,   301,  1948, 18228,  1465, 46238,   277,
         287, 24549, 14290,  1704,   294,   427,   430,   301,   358,
         300,     2,     1,     1,     1,     1,     1,     1,     1,
           1,     1,     1,     1,     1,     1,     1,     1,     1,
           1,     1,     1,     1,     1,     1,     1,     1,     1,
           1,     1,     1,     1,     1,     1,     1,     1,     1,
           1], dtype=int32)

In [ ]:
images[0].shape

(64, 512, 1)

In [ ]:
START_TOKEN=tokenizer.bos_token_id
END_TOKEN=tokenizer.eos_token_id
PADDING_TOKEN=tokenizer.pad_token_id

In [ ]:
print("Special tokens map:")
print(tokenizer.special_tokens_map)

print("\nSpecial token IDs:")
print(tokenizer.special_tokens_map_extended)

Special tokens map:
{'bos_token': '<s>', 'eos_token': '</s>', 'unk_token': '<unk>', 'sep_token': '</s>', 'pad_token': '<pad>', 'cls_token': '<s>', 'mask_token': '<mask>'}

Special token IDs:
{'bos_token': AddedToken("<s>", rstrip=False, lstrip=False, single_word=False, normalized=True, special=True), 'eos_token': AddedToken("</s>", rstrip=False, lstrip=False, single_word=False, normalized=True, special=True), 'unk_token': AddedToken("<unk>", rstrip=False, lstrip=False, single_word=False, normalized=True, special=True), 'sep_token': AddedToken("</s>", rstrip=False, lstrip=False, single_word=False, normalized=True, special=True), 'pad_token': AddedToken("<pad>", rstrip=False, lstrip=False, single_word=False, normalized=True, special=True), 'cls_token': AddedToken("<s>", rstrip=False, lstrip=False, single_word=False, normalized=True, special=True), 'mask_token': AddedToken("<mask>", rstrip=False, lstrip=True, single_word=False, normalized=True, special=True)}


In [ ]:
print("BOS (start) token:", tokenizer.bos_token, tokenizer.bos_token_id)
print("EOS (end) token:", tokenizer.eos_token, tokenizer.eos_token_id)
print("PAD token:", tokenizer.pad_token, tokenizer.pad_token_id)
print("MASK token:", tokenizer.mask_token, tokenizer.mask_token_id)
print("UNK token:", tokenizer.unk_token, tokenizer.unk_token_id)


BOS (start) token: <s> 0
EOS (end) token: </s> 2
PAD token: <pad> 1
MASK token: <mask> 4
UNK token: <unk> 3


In [ ]:
# ============= CONFIGURATION =============
CONFIG = {
    # Model Architecture
    'conv_filters': [32, 64, 96, 128],
    'kernel_size': (3, 3),
    'embedding_dim': 256,
    'decoder_units': 64,
    'attention_units': 64,
    'dropout_rate': 0.5,
    'vocab_size': vocab_size,


    # Image preprocessing
    'img_height': TARGET_HEIGHT,
    'img_width': TARGET_WIDTH,
    'normalize': True,
    # Model
    'batch_size': 2,
    'epochs': 80,
    'initial_lr': 0.001,
    'gradient_clip': 4.0,
    'teacher_forcing_decay': 0.02,
}

In [ ]:
class OCREncoder(tf.keras.Model):
    def __init__(self, config=None):
        super(OCREncoder, self).__init__()
        self.confg = config if config else Config()
        filters = self.confg["conv_filters"]
        kernel = self.confg["kernel_size"]
        dropout = self.confg["dropout_rate"]

        # Block 1: 64 filters (cleaned up unused layers)
        self.conv1_1 = layers.Conv2D(filters[0], kernel, padding='same', kernel_initializer='he_normal')
        self.bn1_1 = layers.BatchNormalization()
        self.pool1 = layers.MaxPooling2D((2, 2))
        self.dropout1 = layers.SpatialDropout2D(dropout)

        # Block 2: 128 filters
        self.conv2_1 = layers.Conv2D(filters[1], kernel, padding='same', kernel_initializer='he_normal')
        self.bn2_1 = layers.BatchNormalization()
        self.pool2 = layers.MaxPooling2D((2, 2))
        self.dropout2 = layers.SpatialDropout2D(dropout)

        # Block 3: 256 filters
        self.conv3_1 = layers.Conv2D(filters[2], kernel, padding='same', kernel_initializer='he_normal')
        self.bn3_1 = layers.BatchNormalization()

        # Block 4: 512 filters
        self.conv4_1 = layers.Conv2D(filters[3], kernel, padding='same', kernel_initializer='he_normal')
        self.bn4_2 = layers.BatchNormalization()
        self.pool4 = layers.MaxPooling2D((2, 1))  # Only pool height


        self.permute = layers.Permute((2, 1, 3))  # For sequence: width as time

    def build(self, input_shape):
        super(OCREncoder, self).build(input_shape)

    def call(self, inputs, training=False):
        # Block 1
        x = self.conv1_1(inputs)
        x = layers.Activation('relu')(x)
        x = self.bn1_1(x, training=training)
        x = self.pool1(x)

        # Block 2
        x = self.conv2_1(x)
        x = layers.Activation('relu')(x)
        x = self.bn2_1(x, training=training)
        x = self.pool2(x)

        # Block 3
        x = self.conv3_1(x)
        x = layers.Activation('relu')(x)
        x = self.bn3_1(x, training=training)

        # Block 4
        x = self.conv4_1(x)
        x = self.bn4_2(x, training=training)
        x = layers.Activation('relu')(x)
        x = self.pool4(x)


        # Reshape for sequence (width as time)
        x = self.permute(x)
        b, t, f, c = tf.shape(x)[0], x.shape[1], x.shape[2], x.shape[3]
        x = tf.reshape(x, [b, t, f * c])
        return x


In [ ]:
class BahdanauAttention(tf.keras.layers.Layer):
    def __init__(self, units):
        super(BahdanauAttention, self).__init__()

        self.units = units

        self.W1 = layers.Dense(self.units)
        self.W2 = layers.Dense(self.units)
        self.V = layers.Dense(1)

    def call(self, features, hidden):

        hidden_with_time_axis = tf.expand_dims(hidden, 1)
        score = tf.nn.tanh(self.W1(features) + self.W2(hidden_with_time_axis))
        attention_weights = tf.nn.softmax(self.V(score), axis=1)  # (B, T, 1)

        # Context for each time step (still (B, T, F))
        context_vector = attention_weights * features
        return context_vector, attention_weights

In [ ]:
class Decoder(tf.keras.Model):
    def __init__(self, config):
        super(Decoder, self).__init__()
        self.confg = config if config else Config()

        self.units = self.confg['decoder_units']
        self.dropout = self.confg['dropout_rate']
        self.vocab_size = self.confg['vocab_size']

        self.attention = BahdanauAttention(units=self.confg['attention_units'])
        self.fc_out = layers.Dense(self.vocab_size)
        self.act = layers.Activation('relu')

        self.rnn1 = layers.Bidirectional(layers.LSTM(self.units*2, return_sequences=True, dropout=self.dropout))
        self.rnn2 = layers.Bidirectional(layers.LSTM(self.units, return_sequences=True, dropout=self.dropout))
    def build(self, input_shape):
        super(Decoder, self).build(input_shape)

    def call(self, x, features, hidden, training=True):
        context_vector, attention_weights = self.attention(features, hidden)

        x = self.rnn1(context_vector)
        x = self.rnn2(x)

        x = self.fc_out(x)  # Shape:(Batch, Timestemp, vocab_size)

        state = None
        return x, state, attention_weights
    def initialize_hidden_state(self, batch_size):
        return tf.zeros((batch_size, self.units))

In [ ]:
# # Learning rate schedule
# lr_schedule = tf.keras.optimizers.schedules.ExponentialDecay(
#     initial_learning_rate=CONFIG['initial_lr'],
#     decay_steps=500,
#     decay_rate=0.91,
#     # staircase=True
# )

# # Or use callback
# reduce_lr = tf.keras.callbacks.ReduceLROnPlateau(
#     monitor='loss',
#     factor=0.4,
#     patience=3,
#     # min_lr=1e-6,
#     verbose=1
# )

In [ ]:
# Initialize models
encoder = OCREncoder(CONFIG)
decoder = Decoder(CONFIG)

# optimizer = tf.keras.optimizers.Adam(learning_rate=lr_schedule)

In [ ]:
encoder.build(input_shape=(None, None, None, None))
decoder.build(input_shape=(None, None, None))

In [ ]:
decoder.summary()

Model: "decoder"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ bahdanau_attention              │ ?                      │   0 (unbuilt) │
│ (BahdanauAttention)             │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation (Activation)         │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_1 (Bidirectional) │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [ ]:
def masked_accuracy(real, pred):
    # real: [batch, seq_len], pred: [batch, seq_len, vocab_size]
    mask = tf.cast(tf.math.not_equal(real, 0), tf.float32)
    pred_ids = tf.argmax(pred, axis=-1, output_type=real.dtype)  # shape: [batch, seq_len]
    correct = tf.cast(tf.equal(real, pred_ids), tf.float32)
    correct *= mask
    return tf.reduce_sum(correct) / tf.reduce_sum(mask)


In [ ]:
loss_object = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)
# loss_object = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True, reduction='none')

def masked_loss(y_true, y_pred, padding_token):
    loss = loss_object(y_true=y_true, y_pred=y_pred)
    mask = tf.cast(tf.not_equal(y_true, padding_token), dtype=loss.dtype)
    loss = loss * mask
    return tf.reduce_sum(loss) / tf.reduce_sum(mask)

In [ ]:
# # ============= TRAINING STEP WITH TEACHER FORCING =============
# # @tf.function
# def train_step(tensor, target, encoder, decoder, optimizer, padding_token, teacher_forcing_ratio=0.5):
#     batch_size = tf.shape(tensor)[0]
#     seq_length = tf.shape(target)[1]

#     with tf.GradientTape() as tape:
#         # Encode
#         features = encoder(tensor, training=False)

#         # Initialize decoder
#         hidden = decoder.initialize_hidden_state(batch_size)
#         # dec_input = tf.expand_dims([0] * batch_size, 1)  # Start token
#         # dec_input = tf.expand_dims([0] * batch_size, 1)
#         dec_input = tf.expand_dims(tf.fill([batch_size], START_TOKEN), 1)

#         loss = 0.0

#         # predictions, hidden, _ = decoder(dec_input, features, hidden, training=True)
#         # loss_ = loss_fn(target, predictions)
#         # mask = tf.cast(tf.not_equal(target, padding_token), dtype=loss_.dtype)
#         # loss = tf.reduce_sum(loss_ * mask) / tf.reduce_sum(mask)
#         # loss += tf.reduce_sum(loss_t * mask)


#         # Decode with teacher forcing
#         for t in range(seq_length):
#             predictions, hidden, _ = decoder(dec_input, features, hidden, training=True)
#             # print(predictions.shape)


#             # Calculate loss for this timestep
#             print("Target:",target[:, t],"Prediction:",np.argmax(predictions[:, 0, :]))
#             loss_t = loss_fn(target[:, t], predictions[:, 0, :])
#             mask = tf.cast(tf.not_equal(target[:, t], padding_token), dtype=loss_t.dtype)
#             loss += tf.reduce_sum(loss_t * mask)

#             # Teacher forcing: use ground truth or prediction
#             # dec_input = tf.expand_dims(target[:, t], 1)
#             use_teacher_forcing = tf.random.uniform([]) < teacher_forcing_ratio
#             dec_input = tf.cond(use_teacher_forcing, lambda:tf.expand_dims(target[:,t], 1), lambda: tf.expand_dims(tf.argmax(predictions[:, 0, :], axis=-1, output_type=tf.int32), 1))
#             # if use_teacher_forcing:
#             #     dec_input = tf.expand_dims(target[:, t], 1)
#             # else:
#             #     predicted_id = tf.argmax(predictions[:, 0, :], axis=-1, output_type=tf.int32)
#             #     dec_input = tf.expand_dims(predicted_id, 1)

#         # Average loss
#         total_mask = tf.cast(tf.not_equal(target, padding_token), dtype=tf.float32)
#         loss = loss / tf.reduce_sum(total_mask)

#         # lossreg = total_loss / (total_mask + 1e-8)

#         # # Add regularization loss
#         # reg_loss = tf.add_n([tf.nn.l2_loss(v) for v in encoder.trainable_variables
#         #                      if 'kernel' in v.name]) * 1e-5

#     # Compute and apply gradients with clipping
#     trainable_vars = encoder.trainable_variables + decoder.trainable_variables
#     gradients = tape.gradient(loss, trainable_vars)
#     # gradients, _ = tf.clip_by_global_norm(gradients, CONFIG['gradient_clip'])
#     optimizer.apply_gradients(zip(gradients, trainable_vars))

#     return loss

In [ ]:
@tf.function
def train_step(tensor, target, encoder, decoder, optimizer, padding_token, teacher_forcing_ratio=0.5):
    batch_size = tf.shape(tensor)[0]

    with tf.GradientTape() as tape:
        features = encoder(tensor, training=False)
        hidden = decoder.initialize_hidden_state(batch_size)
        dec_input = tf.expand_dims(tf.fill([batch_size], tokenizer.bos_token_id), 1)

        predictions, hidden, _ = decoder(dec_input, features, hidden, training=True)
        loss = masked_loss(target, predictions, padding_token)
        # loss_ = loss_fn(target, predictions)
        # mask = tf.cast(tf.not_equal(target, padding_token), dtype=loss_.dtype)
        # loss = tf.reduce_sum(loss_ * mask) / tf.reduce_sum(mask)

    trainable_vars = encoder.trainable_variables + decoder.trainable_variables
    gradients = tape.gradient(loss, trainable_vars)
    gradients, _ = tf.clip_by_global_norm(gradients, 1.0)
    optimizer.apply_gradients(zip(gradients, trainable_vars))
    acc = masked_accuracy(target, predictions)
    print("Accuracy:",acc)
    return loss


In [ ]:
def inference(image, encoder, decoder, max_length, start_token=0, end_token=vocab_size, padding_token=vocab_size):
    image = tf.expand_dims(image, 0) if len(image.shape) == 3 else tf.expand_dims(image, 0)  # Add batch dim
    image = tf.convert_to_tensor(image, dtype=tf.float32)

    features = encoder(image, training=False)
    batch_size = tf.shape(image)[0]

    hidden = decoder.initialize_hidden_state(batch_size)

    # Prepare decoder input (could be start tokens or a fixed tensor as your decoder needs)
    dec_input = tf.expand_dims(tf.fill([batch_size], start_token), 1)

    # Predict the whole sequence at once
    predictions, hidden, attention_weights = decoder(dec_input, features, hidden, training=False)
    # predictions shape: (batch_size, seq_len, vocab_size)

    predicted_ids = tf.argmax(predictions, axis=-1).numpy()  # (batch_size, seq_len)

    return predicted_ids[0], attention_weights.numpy()  # return first batch sample


In [ ]:
# # ============= INFERENCE FUNCTION =============
# def inference(image, encoder, decoder, max_length, start_token=START_TOKEN, end_token=END_TOKEN, padding_token=PADDING_TOKEN):
#     """
#     Arguments Required:
#         image: Input image tensor (H, W, 1) or (H, W)
#         encoder: Trained encoder model
#         decoder: Trained decoder model
#         max_length: Maximum sequence length
#         start_token: Token ID for sequence start
#         end_token: Token ID for sequence end
#         padding_token: Token ID for padding

#     Returns Arguments:
#         predicted_sequence: List of predicted token IDs
#         attention_plot: Attention weights for visualization
#     """
#     # Add batch dimension if needed
#     if len(image.shape) == 2:
#         image = image[..., np.newaxis]
#     if len(image.shape) == 3:
#         image = np.expand_dims(image, 0)

#     image = tf.convert_to_tensor(image, dtype=tf.float32)

#     # Encode
#     features = encoder(image, training=False)

#     # Initialize decoder
#     hidden = decoder.initialize_hidden_state(1)
#     dec_input = tf.expand_dims([start_token], 1)

#     result = []
#     attention_weights_list = []

#     for t in range(max_length):
#         predictions, hidden, attention_weights = decoder(dec_input, features, hidden, training=False)
#         attention_weights_list.append(attention_weights.numpy())

#         # Get predicted token
#         predicted_id = tf.argmax(predictions[0, 0, :]).numpy()
#         result.append(predicted_id)

#         # Stop if end token is predicted
#         if predicted_id == end_token:
#             break

#         # Use prediction as next input
#         dec_input = tf.expand_dims([predicted_id], 1)

#     attention_plot = np.concatenate(attention_weights_list, axis=1)
#     return result, attention_plot

In [ ]:
# # ============= ACCURACY CALCULATION =============
# def calculate_accuracy(predictions, labels, padding_token):

#     total_chars = 0
#     correct_chars = 0
#     total_seqs = 0
#     correct_seqs = 0

#     for pred, label in zip(predictions, labels):
#         # Remove padding from label
#         label_no_pad = [l for l in label if l != padding_token]

#         # Compare sequences
#         seq_match = True
#         for i, l in enumerate(label_no_pad):
#             if i < len(pred):
#                 if pred[i] == l:
#                     correct_chars += 1
#                 else:
#                     seq_match = False
#             else:
#                 seq_match = False
#             total_chars += 1

#         # Check if lengths match
#         if len(pred) != len(label_no_pad):
#             seq_match = False

#         if seq_match:
#             correct_seqs += 1
#         total_seqs += 1

#     char_acc = correct_chars / total_chars if total_chars > 0 else 0
#     seq_acc = correct_seqs / total_seqs if total_seqs > 0 else 0

#     return char_acc, seq_acc

In [ ]:
def calculate_accuracy(predictions, labels, padding_token):
    total_chars = 0
    correct_chars = 0
    total_seqs = 0
    correct_seqs = 0

    for pred, label in zip(predictions, labels):
        # Remove padding from label
        label_no_pad = [l for l in label if l != padding_token]

        # Optionally, remove padding from prediction as well
        pred_no_pad = [p for p in pred if p != padding_token]

        # Character accuracy: count matches up to max length of label and pred
        max_len = max(len(label_no_pad), len(pred_no_pad))
        for i in range(max_len):
            l = label_no_pad[i] if i < len(label_no_pad) else None
            p = pred_no_pad[i] if i < len(pred_no_pad) else None
            if l is not None:
                total_chars += 1
                if p == l:
                    correct_chars += 1

        # Sequence accuracy: exact match including length and content (excluding padding)
        seq_match = (label_no_pad == pred_no_pad)
        if seq_match:
            correct_seqs += 1
        total_seqs += 1

    char_acc = correct_chars / total_chars if total_chars > 0 else 0
    seq_acc = correct_seqs / total_seqs if total_seqs > 0 else 0

    return char_acc, seq_acc


In [ ]:
# ============= VALIDATION FUNCTION =============
def validate(images, labels, encoder, decoder, max_length, padding_token, num_samples=None):

    if num_samples is None:
        num_samples = len(images)

    predictions = []

    for i in range(min(num_samples, len(images))):
        img = images[i]
        pred, _ = inference(img, encoder, decoder, max_length,
                           start_token=tokenizer.bos_token_id, end_token=tokenizer.eos_token_id, padding_token=tokenizer.pad_token_id)
        predictions.append(pred)

    char_acc, seq_acc = calculate_accuracy(predictions[:num_samples],
                                           labels[:num_samples],
                                           tokenizer.pad_token_id)

    return char_acc, seq_acc, predictions

In [ ]:
import os

# ============= MAIN TRAINING LOOP =============
def train_model(images, labels, encoder, decoder, optimizer,
                epochs, batch_size, padding_token, max_length,
                val_images=None, val_labels=None, checkpoint_dir='best_weights'):
    checkpoint = tf.train.Checkpoint(optimizer=optimizer,
                                 encoder=encoder,
                                 decoder=decoder)
    manager = tf.train.CheckpointManager(checkpoint, checkpoint_dir, max_to_keep=3)
    if manager.latest_checkpoint:
        print(f"Restoring from checkpoint: {manager.latest_checkpoint}")
        checkpoint.restore(manager.latest_checkpoint).expect_partial()
    else:
        print("No checkpoint found, starting training from scratch.")

    prev_acc = 0.00  # Track the previous best accuracy
    teacher_forcing_ratio = 0.8

    # Ensure the checkpoint directory exists
    os.makedirs(checkpoint_dir, exist_ok=True)

    for epoch in range(epochs):
        print(f"\n{'='*60}")
        print(f"Epoch {epoch+1}/{epochs}")
        print(f"Teacher Forcing Ratio: {teacher_forcing_ratio:.3f}")
        print(f"{'='*60}")

        # Shuffle data
        num_samples = len(images)
        indices = np.arange(num_samples)
        np.random.shuffle(indices)
        images_shuffled = images[indices]
        labels_shuffled = labels[indices]

        # Training
        epoch_loss = 0
        num_batches = 0

        for i in range(0, num_samples, batch_size):
            batch_images = images_shuffled[i:i+batch_size]
            batch_labels = labels_shuffled[i:i+batch_size]

            # Add channel dimension if needed
            if len(batch_images.shape) == 3:
                batch_images = batch_images[..., np.newaxis]

            # Convert to tensors
            batch_images = tf.convert_to_tensor(batch_images, dtype=tf.float32)
            batch_labels = tf.convert_to_tensor(batch_labels, dtype=tf.int32)

            # Train step
            loss = train_step(batch_images, batch_labels, encoder, decoder,
                            optimizer, padding_token, teacher_forcing_ratio)

            epoch_loss += loss.numpy()
            num_batches += 1

            # Progress
            if (num_batches % 10) == 0:
                print(f"\033[92mBatch {num_batches}/{num_samples//batch_size} | Loss: {loss.numpy():.4f}\033[0m", end='\r')


        avg_loss = epoch_loss / num_batches
        print(f"\nTraining Loss: {avg_loss:.4f}")
        # mlflow.log_metric("train_loss", float(avg_loss), step=epoch)

        # Validation
        if val_images is not None and val_labels is not None:
            print("\nValidating...")
            char_acc, seq_acc, sample_preds = validate(
                val_images, val_labels, encoder, decoder,
                max_length, padding_token, num_samples=min(100, len(val_images))
            )

            print(f"Validation - Char Accuracy: \033[92m{char_acc*100:.2f}%\033[0m | "
                  f"Seq Accuracy: \033[92m{seq_acc*100:.2f}%\033[0m")

            # Show sample predictions
            print("\nSample Predictions:")
            for j in range(min(3, len(sample_preds))):
                pred = sample_preds[j]
                label = [l for l in val_labels[j] if l != padding_token]
                pred = [l for l in sample_preds[j] if l != padding_token]
                print(f"  True: {label}")
                print(f"  Pred: {pred}")
                print("="*10)

            # Check if current seq_acc is better than the previous one
            if char_acc > prev_acc:
                # If improvement, save the model and update prev_acc
                print(f"New best validation accuracy! Saving model...")
                # encoder.save_weights(os.path.join(checkpoint_dir, 'best_encoder.ckpt'))
                # decoder.save_weights(os.path.join(checkpoint_dir, 'best_decoder.ckpt'))
                prev_acc = char_acc  # Update the previous accuracy with the current one
                saved_path = manager.save()
                print("Saved checkpoint for epoch {}: {}".format(int(epoch), saved_path))
            else:
                # If no improvement, restore the weights from the previous best epoch
                print(f"No improvement in validation accuracy. So No Saving New weights")
                # print(f"No improvement in validation accuracy. Restoring best weights from last epoch.")
                # encoder.load_weights(os.path.join(checkpoint_dir, 'best_encoder.weights.h5'))
                # decoder.load_weights(os.path.join(checkpoint_dir, 'best_decoder.weights.h5'))

        # Decay teacher forcing
        if epoch < 10:
            teacher_forcing_ratio = 0.5
        else:
            teacher_forcing_ratio = max(0.1, teacher_forcing_ratio - CONFIG['teacher_forcing_decay'])

        print(f"{'='*60}")


In [ ]:
if __name__ == "__main__":

    # import mlflow
    # import mlflow.tensorflow

    # mlflow.set_experiment("Image_Captioning_Seq2Seq")

    # with mlflow.start_run(run_name="custom_tf_training"):
    #     # Log parameters
    #     mlflow.log_params({
    #         "epochs": CONFIG['epochs'],
    #         "batch_size": CONFIG['batch_size'],
    #         "initial_lr": CONFIG['initial_lr'],
    #     })

        # Optimizer with learning rate schedule
        lr_schedule = tf.keras.optimizers.schedules.ExponentialDecay(
            initial_learning_rate=1e-3,
            decay_steps=10000,
            decay_rate=0.90
        )
        # optimizer = tf.keras.optimizers.Adam(learning_rate=lr_schedule)

        # optimizer = tf.keras.optimizers.AdamW(1e-2, weight_decay=1e-3)
        # optimizer = tf.keras.optimizers.Adam(learning_rate=0.0001)#(learning_rate=lr_schedule)
        optimizer = tf.keras.optimizers.Adam(learning_rate=lr_schedule)

        # Split data into train/val (80/20)
        split_idx = int(0.8 * len(images))
        train_images, val_images = images[:split_idx], images[split_idx:]
        train_labels, val_labels = labels[:split_idx], labels[split_idx:]

        # Train
        train_model(
            images=train_images,
            labels=train_labels,
            encoder=encoder,
            decoder=decoder,
            optimizer=optimizer,
            epochs=CONFIG['epochs'],
            batch_size=CONFIG['batch_size'],
            padding_token=PADDING_TOKEN,
            max_length=MAX_LABEL_LEN,
            val_images=val_images,
            val_labels=val_labels
        )

        print("\nTraining complete!")

        # ============= TEST INFERENCE =============
        print("\nTesting inference on a sample image...")
        test_img = val_images[0]
        predicted_seq, attention = inference(
            test_img, encoder, decoder, MAX_LABEL_LEN,
            start_token=START_TOKEN, end_token=END_TOKEN, padding_token=PADDING_TOKEN
        )

        true_label = [l for l in val_labels[0] if l != PADDING_TOKEN]
        predicted_seq = [l for l in predicted_seq if l != PADDING_TOKEN]
        print(f"True sequence: {true_label}")
        print(f"Predicted sequence: {predicted_seq}")
        # mlflow.tensorflow.log_model(tf.keras.models.Model(), artifact_path="model")

No checkpoint found, starting training from scratch.

Epoch 1/80
Teacher Forcing Ratio: 0.800
Accuracy: Tensor("truediv_1:0", shape=(), dtype=float32)
Accuracy: Tensor("truediv_1:0", shape=(), dtype=float32)


In [ ]:
import tensorflow as tf
import numpy as np

def debug_model_setup(encoder, decoder, images, labels, num_to_char, padding_token):
    print("\n" + "="*70)
    print("MODEL DEBUGGING REPORT")
    print("="*70)

    # 1. Check data statistics
    print("\n1. DATA STATISTICS")
    print("-" * 70)
    print(f"Number of samples: {len(images)}")
    print(f"Image shape: {images.shape}")
    print(f"Label shape: {labels.shape}")
    print(f"Image range: [{images.min():.3f}, {images.max():.3f}]")
    print(f"Label range: [{labels.min()}, {labels.max()}]")

    # Check label distribution
    label_flat = labels.flatten()
    label_flat = label_flat[label_flat != padding_token]
    unique, counts = np.unique(label_flat, return_counts=True)

    print(f"\nLabel distribution (excluding padding):")
    total = len(label_flat)
    for token, count in sorted(zip(unique, counts), key=lambda x: -x[1])[:5]:
        char = num_to_char.get(token, '?')
        percentage = (count / total) * 100
        print(f"  Token {token} ('{char}'): {count:5d} ({percentage:5.1f}%)")

    # Check if heavily imbalanced
    max_percentage = (counts.max() / total) * 100
    if max_percentage > 30:
        print(f"\n⚠️  WARNING: Class imbalance detected! Token {unique[counts.argmax()]} "
              f"appears {max_percentage:.1f}% of the time")
        print("   This can cause mode collapse. Consider class weighting.")

    # 2. Check model architecture
    print("\n2. MODEL ARCHITECTURE")
    print("-" * 70)

    # Test encoder forward pass
    test_image = images[0:1]
    test_image = tf.convert_to_tensor(test_image, dtype=tf.float32)

    try:
        features = encoder(test_image, training=False)
        print(f"✓ Encoder forward pass successful")
        print(f"  Input shape:  {test_image.shape}")
        print(f"  Output shape: {features.shape}")
        print(f"  Output range: [{features.numpy().min():.3f}, {features.numpy().max():.3f}]")

        if features.numpy().std() < 0.01:
            print(f"  ⚠️  WARNING: Very low variance in encoder output!")
            print(f"     This suggests encoder is not learning features.")
    except Exception as e:
        print(f"✗ Encoder forward pass FAILED: {str(e)}")
        return False

    # Test decoder forward pass
    batch_size = 1
    hidden = decoder.initialize_hidden_state(batch_size)
    dec_input = tf.zeros((batch_size, 1), dtype=tf.int32)

    try:
        predictions, new_hidden, attn = decoder(dec_input, features, hidden, training=False)
        print(f"\n✓ Decoder forward pass successful")
        print(f"  Feature shape:    {features.shape}")
        print(f"  Hidden shape:     {hidden.shape}")
        print(f"  Prediction shape: {predictions.shape}")
        print(f"  Attention shape:  {attn.shape}")

        # Check prediction distribution
        probs = tf.nn.softmax(predictions[0, 0, :]).numpy()
        top_5_indices = np.argsort(probs)[-5:][::-1]
        print(f"\n  Top 5 predictions (before training):")
        for idx in top_5_indices:
            char = num_to_char.get(idx, '?')
            print(f"    Token {idx} ('{char}'): {probs[idx]*100:.1f}%")

        if probs.max() > 0.9:
            print(f"  ⚠️  WARNING: Model already very confident before training!")
            print(f"     Max probability: {probs.max()*100:.1f}%")
            print(f"     This suggests poor initialization.")
    except Exception as e:
        print(f"✗ Decoder forward pass FAILED: {str(e)}")
        return False

    # 3. Check gradients
    print("\n3. GRADIENT CHECK")
    print("-" * 70)

    batch_images = tf.convert_to_tensor(images[0:2], dtype=tf.float32)
    batch_labels = tf.convert_to_tensor(labels[0:2], dtype=tf.int32)

    with tf.GradientTape() as tape:
        features = encoder(batch_images, training=True)
        hidden = decoder.initialize_hidden_state(2)
        dec_input = tf.expand_dims([0, 0], 1)

        predictions, _, _ = decoder(dec_input, features, hidden, training=True)

        # Simple loss
        loss = tf.nn.sparse_softmax_cross_entropy_with_logits(
            labels=batch_labels[:, 0],
            logits=predictions[:, 0, :]
        )
        loss = tf.reduce_mean(loss)

    trainable_vars = encoder.trainable_variables + decoder.trainable_variables
    gradients = tape.gradient(loss, trainable_vars)

    # Check for None gradients
    none_grads = sum(1 for g in gradients if g is None)
    if none_grads > 0:
        print(f"✗ {none_grads} variables have None gradients!")
        return False

    # Check gradient magnitudes
    grad_magnitudes = [tf.reduce_mean(tf.abs(g)).numpy() for g in gradients]

    print(f"✓ All gradients computed successfully")
    print(f"  Mean gradient magnitude: {np.mean(grad_magnitudes):.6f}")
    print(f"  Max gradient magnitude:  {np.max(grad_magnitudes):.6f}")
    print(f"  Min gradient magnitude:  {np.min(grad_magnitudes):.6f}")

    if np.mean(grad_magnitudes) < 1e-8:
        print(f"  ⚠️  WARNING: Gradients are extremely small!")
        print(f"     Model may not learn effectively.")

    if np.max(grad_magnitudes) > 100:
        print(f"  ⚠️  WARNING: Some gradients are very large!")
        print(f"     Consider gradient clipping or lower learning rate.")

    # Show gradient distribution for key layers
    print(f"\n  Gradient magnitudes by layer (first 5):")
    for var, grad_mag in list(zip(trainable_vars, grad_magnitudes))[:5]:
        print(f"    {var.name[:50]:50s} {grad_mag:.6f}")

    # 4. Check for common issues
    print("\n4. COMMON ISSUES CHECK")
    print("-" * 70)

    issues_found = []

    # Check 1: Image normalization
    if images.max() > 10:
        issues_found.append("Images not normalized (max > 10)")

    # Check 2: Label token range
    if labels.max() >= len(num_to_char):
        issues_found.append(f"Label tokens exceed vocab size ({labels.max()} >= {len(num_to_char)})")

    # Check 3: Sequence lengths
    seq_lengths = np.sum(labels != padding_token, axis=1)
    if seq_lengths.max() >= labels.shape[1]:
        issues_found.append("Some sequences exceed max length")

    # Check 4: Empty sequences
    if seq_lengths.min() <= 2:  # Only START and END
        issues_found.append("Some sequences are empty or too short")

    if issues_found:
        print("✗ Issues found:")
        for issue in issues_found:
            print(f"  - {issue}")
    else:
        print("✓ No obvious issues detected")

    # 5. Recommendations
    print("\n5. RECOMMENDATIONS")
    print("-" * 70)

    recommendations = []

    # Based on gradient magnitude
    if np.mean(grad_magnitudes) < 1e-6:
        recommendations.append("• Use higher learning rate (try 0.001)")
    elif np.mean(grad_magnitudes) > 1:
        recommendations.append("• Use lower learning rate (try 0.0001)")

    # Based on class imbalance
    if max_percentage > 30:
        recommendations.append("• Implement class weighting in loss function")
        recommendations.append("• Try focal loss instead of cross-entropy")

    # Based on encoder output
    if features.numpy().std() < 0.1:
        recommendations.append("• Check encoder initialization")
        recommendations.append("• Try removing some dropout layers")

    # Based on decoder confidence
    if probs.max() > 0.9:
        recommendations.append("• Decoder initialization is poor - reinitialize model")
        recommendations.append("• Add label smoothing to loss function")

    if recommendations:
        for rec in recommendations:
            print(rec)
    else:
        print("✓ Model looks ready for training")

    print("\n" + "="*70)
    print("END OF DEBUG REPORT")
    print("="*70 + "\n")

    return len(issues_found) == 0 and len(recommendations) <= 2

In [ ]:
# ============= QUICK FIX SUGGESTIONS =============
def suggest_fixes(encoder, decoder, images, labels, num_to_char, padding_token):
    """
    Analyze issues and suggest specific fixes.
    """
    print("\n" + "="*70)
    print("SUGGESTED FIXES FOR MODE COLLAPSE")
    print("="*70)

    # Analyze current predictions
    print("\n1. Analyzing current model behavior...")

    test_images = tf.convert_to_tensor(images[:5], dtype=tf.float32)
    predictions_list = []

    for i in range(5):
        features = encoder(test_images[i:i+1], training=False)
        hidden = decoder.initialize_hidden_state(1)
        dec_input = tf.zeros((1, 1), dtype=tf.int32)

        pred_sequence = []
        for _ in range(10):  # Predict 10 tokens
            preds, hidden, _ = decoder(dec_input, features, hidden, training=False)
            predicted_id = tf.argmax(preds[0, 0, :]).numpy()
            pred_sequence.append(predicted_id)
            dec_input = tf.expand_dims([predicted_id], 1)

        predictions_list.append(pred_sequence)

    # Check if all predictions are the same
    unique_preds = set(tuple(p) for p in predictions_list)

    if len(unique_preds) == 1:
        most_common = predictions_list[0][0]
        char = num_to_char.get(most_common, '?')
        print(f"   ✗ CRITICAL: All predictions are identical!")
        print(f"     Always predicting token {most_common} ('{char}')")
        print(f"\n   IMMEDIATE FIXES NEEDED:")
        print(f"   1. Reinitialize the model completely")
        print(f"   2. Use lower learning rate: 0.0003")
        print(f"   3. Remove or reduce recurrent_dropout in GRU")
        print(f"   4. Start with teacher_forcing_ratio=1.0 for first 20 epochs")
        print(f"   5. Add label smoothing to the loss (e.g., smoothing=0.1)")
        print(f"   6. Verify loss is decreasing at each step (track moving average)")
    else:
        print(f"   ✓ Model predictions show diversity.")
        print(f"     Number of unique predicted sequences: {len(unique_preds)}")
        print(f"\n   Recommended general checks:")
        print(f"   • Monitor validation loss closely — early signs of collapse")
        print(f"   • Reduce model complexity if overfitting occurs")
        print(f"   • Ensure dropout is not too high (>0.5)")
        print(f"   • Try focal loss if imbalance is confirmed")

    print("\n" + "="*70)
    print("END OF FIX SUGGESTIONS")
    print("="*70 + "\n")

In [ ]:
ok_to_train = debug_model_setup(
    encoder=encoder,
    decoder=decoder,
    images=train_images,
    labels=train_labels,
    num_to_char=num_to_char,
    padding_token=PADDING_TOKEN
)

if not ok_to_train:
    print("\n❌ Issues found during model setup. Please fix them before training.\n")
    exit()

In [ ]:
import os
import cv2
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow.keras import backend as K
from keras.models import Model
from keras.layers import Input, Conv2D, MaxPooling2D, Reshape, Bidirectional, LSTM, Dense, Lambda, Activation, BatchNormalization, Dropout
from keras.optimizers import Adam

In [ ]:
# Input layer: expects grayscale images of shape (256, 64, 1)
input_data = Input(shape=(64, 512, 1), name='input')

# First Convolutional Block
inner = Conv2D(32, (3, 3), padding='same', name='conv1', kernel_initializer='he_normal')(input_data)
inner = BatchNormalization()(inner)
inner = Activation('relu')(inner)
inner = MaxPooling2D(pool_size=(2, 2), name='max1')(inner)

# Second Convolutional Block
inner = Conv2D(64, (3, 3), padding='same', name='conv2', kernel_initializer='he_normal')(inner)  #
inner = BatchNormalization()(inner)
inner = Activation('relu')(inner)
inner = MaxPooling2D(pool_size=(2, 2), name='max2')(inner)
inner = Dropout(0.3)(inner)

# Third Convolutional Block
inner = Conv2D(128, (3, 3), padding='same', name='conv3', kernel_initializer='he_normal')(inner)  # 128 filters, 3x3 kernel
inner = BatchNormalization()(inner)
inner = Activation('relu')(inner)
inner = MaxPooling2D(pool_size=(1, 2), name='max3')(inner)
inner = Dropout(0.3)(inner)

# Reshape layer to prepare for RNN input
inner = Reshape(target_shape=((64, 1024)), name='reshape')(inner)  # Reshape to (64, 1024)
inner = Dense(64, activation='relu', kernel_initializer='he_normal', name='dense1')(inner)

# Recurrent Neural Network (RNN) Layers
inner = Bidirectional(LSTM(256, return_sequences=True), name='lstm1')(inner)  # Bidirectional LSTM with 256 units, returns sequences
inner = Bidirectional(LSTM(256, return_sequences=True), name='lstm2')(inner)  # Another Bidirectional LSTM with 256 units

# Output layer
inner = Dense(vocab_size, kernel_initializer='he_normal', name='dense2')(inner)
y_pred = Activation('softmax', name='softmax')(inner)

# Model definition
model = Model(inputs=input_data, outputs=y_pred)
model.summary()

ValueError: The total size of the tensor must be unchanged. Received: input_shape=(16, 64, 128), target_shape=(64, 1024)

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models

def build_crnn_model(img_width=512, img_height=64, num_classes=150):
    inputs = layers.Input(shape=(img_height, img_width, 1), name="image_input")

    # CNN Feature Extractor
    x = layers.Conv2D(32, (3, 3), activation="relu", padding="same")(inputs)
    x = layers.MaxPooling2D((2, 2))(x) # (32, 256, 32)

    x = layers.Conv2D(64, (3, 3), activation="relu", padding="same")(x)
    x = layers.MaxPooling2D((2, 2))(x) # (16, 128, 64)

    x = layers.Conv2D(128, (3, 3), activation="relu", padding="same")(x)
    x = layers.MaxPooling2D((2, 1))(x) # Reduce height more than width: (8, 128, 128)

    x = layers.Conv2D(256, (3, 3), activation="relu", padding="same")(x)
    x = layers.MaxPooling2D((2, 1))(x) # (4, 128, 256)

    # Reshape to Sequence (Batch, TimeSteps, Features)
    # TimeSteps = 128, Features = 4 * 256 = 1024
    x = layers.Reshape(target_shape=(128, 1024))(x)
    x = layers.Dense(64, activation="relu")(x)

    # Bidirectional RNNs
    x = layers.Bidirectional(layers.LSTM(256, return_sequences=True))(x)
    x = layers.Bidirectional(layers.LSTM(128, return_sequences=True))(x)

    # Output layer (num_classes + 1 for CTC blank)
    outputs = layers.Dense(num_classes + 1, activation="softmax", name="logits")(x)

    return models.Model(inputs=inputs, outputs=outputs)

model = build_crnn_model(img_width=512, img_height=64, num_classes=vocab_size)

In [ ]:
def ctc_lambda_func(args):
    y_pred, labels, input_length, label_length = args
    # the 2 is critical here since the first couple outputs of the RNN
    # tend to be garbage
    y_pred = y_pred[:, 2:, :]
    return K.ctc_batch_cost(labels, y_pred, input_length, label_length)

In [ ]:
labels = Input(name='gtruth_labels', shape=[vocab_size], dtype='float32')
input_length = Input(name='input_length', shape=[1], dtype='int64')
label_length = Input(name='label_length', shape=[1], dtype='int64')

In [ ]:
#                                             Updated
# Your CTC loss function
def ctc_lambda_func(args):
    y_pred, labels, input_length, label_length = args
    y_pred = y_pred[:, 2:, :]  # skip the first 2 time steps
    return K.ctc_batch_cost(labels, y_pred, input_length, label_length)

# Later in your model definition
ctc_loss = Lambda(ctc_lambda_func, output_shape=(1,), name='ctc')([y_pred, labels, input_length, label_length])
model_final = Model(inputs=[input_data, labels, input_length, label_length], outputs=ctc_loss)


ValueError: The name "input" is used 2 times in the model. All operation names should be unique.

In [ ]:
model_final.compile(loss={'ctc': lambda y_true, y_pred: y_pred}, optimizer=Adam(learning_rate = 0.0001))

In [ ]:
t_labels.shape
train_label_len = np.zeros([len(t_labels), 1])
train_input_len = np.ones([len(t_labels), 1]) * (MAX_LABEL_LEN-2)
train_output = np.zeros([len(t_labels)])

In [ ]:
history = model_final.fit(
    x=[images, t_labels, train_input_len, train_label_len],
    y=train_output,
    # validation_data=([valid_x, valid_y, valid_input_len, valid_label_len], valid_output),
    epochs=100,
    batch_size=64
)

Epoch 1/100


ValueError: Input 0 of layer "functional_11" is incompatible with the layer: expected shape=(None, 64, 256, 1), found shape=(None, 64, 512)